# Chain ID standardization

## What is the goal of this notebook?

This is a continuation of the notebook for Step 3.1. 

Sometimes, entities can be mislabeled, and we want to make sure that the chains with the same sequence in different cifs are all named the same way. To make that happen, we do pairwise sequence alignment between all sequences and a reference sequence. In this notebook, we will show how to generate the reference sequences, and how to load it for analysis of new structures. 

>**NOTE:** This is just an example, but we recommend using this tool when dealing with bigger datasets (not just 6 structures as in our dataset). This notebook intends to show you some of the issues you may run into when using this approach. 


## First, import library and setup directories

In [1]:
from CIFcurator import cifcurator_io

In [2]:
PROJDIR="./TIM/"

In [3]:
cifcurator_io.check_project(projdir=PROJDIR, action='create', level='standard_ChainID_bank_2')

### What do you need to generate reference sequences? 

We need a set of structures. We can decide if we want to use all the structures that we have curated so far, or maybe the user would want to use structures coming only from a particular organism (e.g. E. coli). For this tutorial we will use a subset of the structures we have analyzed until now. The next two cells create a new directory and copy only two of the CIFs from our dataset.

In [4]:
cifcurator_io.check_project(projdir=PROJDIR, action='create', level='standard_MolID_bank2')

In [5]:
!cp $PROJDIR/standard_MolID_bank/pdb_00002y62+01.cif  $PROJDIR/standard_MolID_bank2/
!cp $PROJDIR/standard_MolID_bank/pdb_00001ag1+01.cif  $PROJDIR/standard_MolID_bank2/

### How to generate a reference sequence using `CIFcurator_ChainStandardization.py`

We can use `CIFcurator_ChainStandardization.py` to generate the reference sequences. 

In the terminal, type:

> CIFcurator_ChainStandardization.py `{Input Directory}` `{Output Directory}`

Select the following choices when prompted in the on screen menus:  

>`2) Generate Standard Sequences based on all the input structures` -> 
`4) Generate Standard Sequences based on all input structures` -> `QUIT`

The chosen reference sequences will be printed to screen, and saved in JSON format to `{Output Directory}/StandardSequences.json` (here, `./TIM/standard_ChainID_bank_2/StandardSequences.json`).

Not only that, all the possible sequences identfied in our dataset will be also printed, as well as showing how prevalent are those sequences in our structures dataset (see number after "Number of structures:"). The "reference" will be the longest sequence that is repeated the most number of times. 

We suggest to take some time inspecting these results. It can help you identify any outlier that has sift through the previous curation steps. It can also show you how diverse your dataset is.




In [6]:
! echo '2\n4\nQUIT\n' | CIFcurator_ChainStandardization.py $PROJDIR/standard_MolID_bank2 $PROJDIR/standard_ChainID_bank_2

Reading: ./TIM//standard_MolID_bank2/pdb_00002y62+01.cif  (1 of 2)
Reading: ./TIM//standard_MolID_bank2/pdb_00001ag1+01.cif  (2 of 2)
CIFcurator ChainID Standardization Menu
    Select one of the following options to proceed:
    1) Select Standard Sequences from a chosen input structure
    2) Generate Standard Sequences based on all the input structures
Option Number:     Generate Standard Sequences based on all the input structures.
    Type QUIT to return to the main menu.
    1) Show list of chain IDs for Standard Sequences
    2) Enter chain IDs to remove from list
    3) Input file with list of chain IDs to remove
    4) Generate Standard Sequences based on all input structures
    5) Load previously generated standard sequences
Option Number: A
this_chansseq_set
{'AKPQPIAAANWKCNGTTASIEKLVQVFNEHTISHDVQCVVAPTFVHIPLVQAKLRNPKYVISAQNAIAKSGAFTGEVSMPILKDIGVHWVILGHSERRTYYGETDEIVAQKVSEACKQGFMVIACIGETLQQREANQTAKVVLSQTSAIAAKLTKDAWNQVVLAYEPVWAIGTGKVATPEQAQEVHLLLRKWVSENIGTDVAAKLRILYGGSVNAAN

### How to save reference sequence:

The reference sequences are saved automatically in the output directory, in JSON format, so they can be loaded again as they are: `./TIM/standard_ChainID_bank_2/StandardSequences.json`.

This file is rewritten every time the standard sequences are generated, loaded or edited, so we suggest to copy it with a new name before running the script again. The cell below copies it to `./TIM/refseqs.json`. In this step the user can also decide to use a different sequence for any chain, by editing the copy in a text editor (keep the double quotes, as JSON requires them).

>**Note:** Earlier versions of CIFcurator only printed the dictionary to the screen, and the user had to copy it into a text file and change the single quotes to double quotes. This is not needed anymore. Text files made that way can still be loaded.


In [7]:
!cp $PROJDIR/standard_ChainID_bank_2/StandardSequences.json $PROJDIR/refseqs.json

In [8]:
!cat $PROJDIR/refseqs.json

{
    "A": "SKPQPIAAANWKCNGSQQSLSELIDLFNSTSINHDVQCVVASTFVHLAMTKERLSHPKFVIAAQNAIAKSGAFTGEVSLPILKDFGVNWIVLGHSERRAYYGETNEIVADKVAAAVASGFMVIACIGETLQERESGRTAVVVLTQIAAIAKKLKKADWAKVVIAYEPVWAIGTGKVATPQQAQEAHALIRSWVSSKIGADVAGELRILYGGSVNGKNARTLYQQRDVNGFLVGGASLKPEFVDIIKATQ",
    "Ab": "SKPQPIAAANWKCNGSQQSLSELIDLFNSTSINHDVQCVVASTFVHLAMTKERLSHPKFVIAAQNAIAKSGAFTGEVSLPILKDFGVNWIVLGHSERRAYYGETNEIVADKVAAAVASGFMVIACIGETLQERESGRTAVVVLTQIAAIAKKLKKADWAKVVIAYEPVWAIGTGKVATPQQAQEAHALIRSWVSSKIGADVAGELRILYGGSVNGKNARTLYQQRDVNGFLVGGASLKPEFVDIIKATQ"
}


Now we have a file that contains the reference sequences (refseqs.json). The next step will show the user how to load this file. 

### Load reference sequences

Now that we have created a JSON file with the reference sequences, it is time to load them before doing the sequence alignments. 

In the terminal, type:

> CIFcurator_ChainStandardization.py `{Input Directory}` `{Output Directory}`

Select the following choices when prompted in the on screen menus:  

>`2) Generate Standard Sequences based on all the input structures` -> 
`5) Load previously generated standard sequences` -> `./TIM/refseqs.json` -> 
`4) Perform Standardization of Chain IDs` -> `4) Perform pairwise alignments against Standard Sequences and rename chains (if needed)`

**Notice** that even though we created the references with only 2 structures, we are going to use it for our dataset of 6 structures. In practice we would suggest to do this in the reverse way, that means we can create a reference based on a big dataset (do this once), and then run this script with smaller datasets. This way you can run the script in parallel, and speed up this step. It is also useful if you had already curated a dataset, and want to include newly released structures. 


In [9]:
! echo '2\n5\n./TIM/refseqs.json\n4\n4\n' | CIFcurator_ChainStandardization.py $PROJDIR/standard_MolID_bank $PROJDIR/standard_ChainID_bank_2

Reading: ./TIM//standard_MolID_bank/AF-B5YL53-F1+01.cif  (1 of 8)
Reading: ./TIM//standard_MolID_bank/pdb_00002y62+01.cif  (2 of 8)
Reading: ./TIM//standard_MolID_bank/pdb_00001ag1+01.cif  (3 of 8)
Reading: ./TIM//standard_MolID_bank/pdb_00001aw1+04.cif  (4 of 8)
Reading: ./TIM//standard_MolID_bank/pdb_00001aw1+01.cif  (5 of 8)
Reading: ./TIM//standard_MolID_bank/pdb_00001aw1+03.cif  (6 of 8)
Reading: ./TIM//standard_MolID_bank/pdb_00001aw1+02.cif  (7 of 8)
Reading: ./TIM//standard_MolID_bank/AF-B8E2R2-F1+01.cif  (8 of 8)
CIFcurator ChainID Standardization Menu
    Select one of the following options to proceed:
    1) Select Standard Sequences from a chosen input structure
    2) Generate Standard Sequences based on all the input structures
Option Number:     Generate Standard Sequences based on all the input structures.
    Type QUIT to return to the main menu.
    1) Show list of chain IDs for Standard Sequences
    2) Enter chain IDs to remove from list
    3) Input file with list 

### Visualize results

In [10]:
! cat $PROJDIR'/standard_ChainID_bank_2/ChainStandardizationRecord.txt'

./TIM//standard_MolID_bank/AF-B5YL53-F1+01.cif:A:A:0.375968992248062
./TIM//standard_MolID_bank/pdb_00002y62+01.cif:A:A:0.7028112449799196
./TIM//standard_MolID_bank/pdb_00002y62+01.cif:Ab:Ab:0.7028112449799196
./TIM//standard_MolID_bank/pdb_00001ag1+01.cif:A:A:1.0
./TIM//standard_MolID_bank/pdb_00001ag1+01.cif:Ab:Ab:1.0
./TIM//standard_MolID_bank/pdb_00001aw1+04.cif:A:Ab:0.38461538461538464
./TIM//standard_MolID_bank/pdb_00001aw1+04.cif:Ab:A:0.3861003861003861
./TIM//standard_MolID_bank/pdb_00001aw1+01.cif:A:Ab:0.38461538461538464
./TIM//standard_MolID_bank/pdb_00001aw1+01.cif:Ab:A:0.3861003861003861
./TIM//standard_MolID_bank/pdb_00001aw1+03.cif:A:Ab:0.38461538461538464
./TIM//standard_MolID_bank/pdb_00001aw1+03.cif:Ab:A:0.3861003861003861
./TIM//standard_MolID_bank/pdb_00001aw1+02.cif:A:Ab:0.38461538461538464
./TIM//standard_MolID_bank/pdb_00001aw1+02.cif:Ab:A:0.3861003861003861
./TIM//standard_MolID_bank/AF-B8E2R2-F1+01.cif:A:A:0.3764705882352941


### Final comments on this example

We can compare with the results from our previous notebook:
    

In [11]:
! cat $PROJDIR'/standard_ChainID_bank/ChainStandardizationRecord.txt'

./TIM/standard_MolID_bank/AF-B5YL53-F1+01.cif:A:Ab:0.3735408560311284
./TIM/standard_MolID_bank/pdb_00002y62+01.cif:A:Ab:0.3861003861003861
./TIM/standard_MolID_bank/pdb_00002y62+01.cif:Ab:A:0.38461538461538464
./TIM/standard_MolID_bank/pdb_00001ag1+01.cif:A:Ab:0.3861003861003861
./TIM/standard_MolID_bank/pdb_00001ag1+01.cif:Ab:A:0.38461538461538464
./TIM/standard_MolID_bank/pdb_00001aw1+04.cif:A:A:1.0
./TIM/standard_MolID_bank/pdb_00001aw1+04.cif:Ab:Ab:1.0
./TIM/standard_MolID_bank/pdb_00001aw1+01.cif:A:A:1.0
./TIM/standard_MolID_bank/pdb_00001aw1+01.cif:Ab:Ab:1.0
./TIM/standard_MolID_bank/pdb_00001aw1+03.cif:A:A:1.0
./TIM/standard_MolID_bank/pdb_00001aw1+03.cif:Ab:Ab:1.0
./TIM/standard_MolID_bank/pdb_00001aw1+02.cif:A:A:1.0
./TIM/standard_MolID_bank/pdb_00001aw1+02.cif:Ab:Ab:1.0
./TIM/standard_MolID_bank/AF-B8E2R2-F1+01.cif:A:Ab:0.37109375


We can see that the results from our previous notebook are better, as we were actually using a sequence that was the most representative of the dataset we intended to analyze. Our example is also an extreme case, as TIM is a homodimer, so either chain could be named A or B, and the assignment will depend on small differences among chains (maybe some residues missing in one of the chains). We want to highlight the importance of knowing your system, the dataset you are using, and what your motivation is when curating these structures. 

Also, as mentioned previously, we suggest to use this tool to get the reference sequences based on the complete data set, and then run this step in batches, in parallel, to speed up this step. 

In some cases, you may also not want to include certain chains in the analysis. For example, if only one or two structure have a particular chain, you may want to remove such chain from the analysis, as this will increase the speed of this process significantly. You only need to provide a text file, where in each line you write the name of the chain you want to exclude. Provide the file name when the following option is prompted:

>`3) Input file with list of chain IDs to add to ignore list` 

